# 05_03 - Tail Risk Model: Upper-Tail Exceedances and Stress Days
## 1. Methodology Overview

This notebook focuses on the upper tail of the forecast distribution and on the days where realized prices exceed the model's risk signal.
The relevant project sources are `src/models/evaluate_model.py`, `src/pipeline/run_backtest.py`, and the backtest summaries under `data/outputs/backtests/`.

**Source artifacts used in this notebook:**
- `data/outputs/backtests/quantile_coverage_summary.csv`
- `data/outputs/backtests/quantile_upper_tail_exceedance_summary.csv`
- `data/outputs/backtests/extreme_days_vs_spot_only.csv`

The purpose is to read the real tail-risk outputs generated by the project's pipeline and inspect the days that matter most for procurement resilience.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
if not (project_root / 'src').exists():
    project_root = Path('../../').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

backtests_dir = Path('../../data/outputs/backtests')
coverage_path = backtests_dir / 'quantile_coverage_summary.csv'
tail_path = backtests_dir / 'quantile_upper_tail_exceedance_summary.csv'
extreme_days_path = backtests_dir / 'extreme_days_vs_spot_only.csv'

coverage_df = pd.read_csv(coverage_path)
tail_df = pd.read_csv(tail_path)
extreme_days_df = pd.read_csv(extreme_days_path)

print(f'Loaded coverage summary from: {coverage_path}')
print(f'Loaded upper-tail summary from: {tail_path}')
print(f'Loaded extreme-day comparison from: {extreme_days_path}')

display(coverage_df)
display(tail_df)
display(extreme_days_df.head())

Loaded coverage summary from: ../../data/outputs/backtests/quantile_coverage_summary.csv
Loaded upper-tail summary from: ../../data/outputs/backtests/quantile_upper_tail_exceedance_summary.csv
Loaded extreme-day comparison from: ../../data/outputs/backtests/extreme_days_vs_spot_only.csv


,quantile,empirical_coverage,coverage_error,n_obs
0,0.50,0.480712,-0.019288,337.0
1,0.90,0.703264,-0.196736,337.0
2,0.95,0.756677,-0.193323,337.0


,quantile,empirical_exceedance_rate,target_exceedance_rate,exceedance_error,n_obs
0,0.50,0.519288,0.50,0.019288,337.0
1,0.90,0.296736,0.10,0.196736,337.0
2,0.95,0.243323,0.05,0.193323,337.0


,date,total_cost_spot_only,reference_extreme_threshold,total_cost_static_hedge,cost_difference_static_hedge_vs_spot_only,total_cost_heuristic_policy,cost_difference_heuristic_policy_vs_spot_only,total_cost_rl_policy,cost_difference_rl_policy_vs_spot_only
0,2024-08-29,120.37,115.986,99.216,-21.154,90.15,-30.22,2.00,-118.37
1,2024-08-31,116.61,115.986,98.858,-17.752,91.25,-25.36,116.61,0.00
2,2024-09-02,116.66,115.986,95.373,-21.287,86.25,-30.41,2.00,-114.66
3,2024-11-05,117.05,115.986,92.767,-24.283,82.36,-34.69,2.00,-115.05
4,2024-11-06,116.83,115.986,92.239,-24.591,81.70,-35.13,2.00,-114.83


## 2. Recomputing Tail Exceedances From the Real Validation Data

To keep the notebook fully reproducible, we also recompute the upper-tail exceedance days directly from the project's validation split and quantile model wrapper.
This uses the same code path as the pipeline, not a synthetic example.

In [2]:
from src.models.evaluate_model import combine_quantile_predictions
from src.models.train_model import train_quantile_suite

train_df = pd.read_csv('../../data/processed/train_features.csv')
validation_df = pd.read_csv('../../data/processed/validation_features.csv')
quantile_output = train_quantile_suite(train_df=train_df, eval_df=validation_df)
prediction_frame = combine_quantile_predictions(quantile_output.results)
prediction_frame['q90_exceedance'] = prediction_frame['y_true'] > prediction_frame['q_0.9']

exceedance_count = int(prediction_frame['q90_exceedance'].sum())
exceedance_share = exceedance_count / len(prediction_frame)
print(f'Upper-tail exceedances: {exceedance_count} of {len(prediction_frame)} validation rows')
print(f'Exceedance share: {exceedance_share:.3%}')

display(prediction_frame.loc[prediction_frame['q90_exceedance']].sort_values('y_true', ascending=False).head(10))

Upper-tail exceedances: 100 of 337 validation rows
Exceedance share: 29.674%


,y_true,q_0.5,q_0.9,q_0.95,q90_exceedance
345,146.67,104.539686,119.783542,121.222382,True
346,143.73,102.969253,137.106297,137.106297,True
329,143.25,92.189730,112.983118,112.983118,True
335,143.21,109.408316,122.556800,122.836941,True
363,140.94,125.153478,125.153478,125.153478,True
336,140.43,111.065866,136.846688,138.206654,True
344,139.09,102.290724,125.925452,125.925452,True
330,136.45,107.460452,120.198915,123.585897,True
321,136.37,102.973948,110.874462,113.359464,True
362,136.15,106.306118,121.519071,121.519071,True


## 3. Interpretation

The tail-risk model is not meant to replace the quantile model; it is a diagnostic layer that tells us when the upper tail is systematically breached and where the policy should be most cautious.
Those stress days are the ones the backtesting layer later compares across strategies.

## 4. From Tail Risk to Procurement Decisions

Identifying an upper-tail exceedance day is only half the job.
The question for the factory manager is: **what do I do about it?**

The DSS maps tail-risk signals into the action catalog:

| Tail-risk level | Signal | Recommended action |
|---|---|---|
| Low (q90 - M1 < 5) | Normal dispersion | `do_nothing` — spot is acceptable |
| Medium (q90 - M1 in 5-12) | Elevated but contained | `buy_m1_future` — lock in front month |
| High (q90 - M1 > 12) | Extended tail | `buy_m2_future` — hedge further out |
| Extreme (q90 - M1 > 18) | Stress regime | `buy_m3_future` + possibly `decrease_production` |

These thresholds are configured in `src/config/constants.py` and can be adjusted
via the sensitivity analysis in notebook 07_03.

## 5. Coverage of the q90 Model

A well-calibrated q90 model should be **exceeded by realized prices ~10% of the time**.
If coverage exceeds 10%, the model is too conservative (overestimates tail risk).
If it falls below 10%, the model underestimates risk — dangerous for procurement decisions.

Coverage is computed in `src/models/evaluate_model.py::evaluate_quantile_coverage()`.

The tail risk model adds value precisely when it is **well-calibrated**:
- On normal days: q90 correctly gives a wide but rarely breached upper bound.
- On stress days: q90 rises ahead of actual price spikes, giving the policy time to hedge.